In [ ]:
using Pkg
Pkg.activate(@__DIR__)
Pkg.status() 
Pkg.instantiate()
using Revise, Distributions, GLMakie
using TrypColonies

### Parameter set 1 
1. **Small exploratory system** — 5 agents on a 200×200 grid (6×6 mm), moving at 3×10⁻⁵ m/s, which is considerably faster than real trypanosomes. This set is intended to build intuition for the effect of the various parameters without long computation times. Agent arrows are rendered (`draw_arrows = true`) so individual trajectories are visible.
'

In [ ]:
para_phys = parameters_physical(
    N                       = (200,200),
    L                       = (0.006,0.006),
    scale_fac               = 4,
    total_time              = 600.0,
    walker_speed_phy        = 3*10^-5,
    Diff_coe                = 7*10^-11,
    walker_step_size        = 5,
    adsorption_rate         = .5,
    decay_rate              = 0.0001,
    Diameter_colony         = 0.004,
    agent_number            = 5,
    noise_strength          = 0.2,
    grid_strength           = 2,  
    grid_recover_rate       = 0.1,
    radius_collision        = 10,
    radius_tanget           = 8,
    
    Δt_walker_min           = 0.1,  #alpha
    growth_rate             = .001
    )

para = parameters( 
    pa_ph                   = para_phys,
    chemotaxis_flag         = true,
    repulsion_flag          = false,
    repulsion_range         = 2,
    geometry                = "circle",
    start_config_gradient   = "random",
    )
    
plot_para = plot_parameters(
    arrow_tip_size   = 6,
    arrow_tail_length= 0.00012,
    framerate        = 30, 
    fontsize         = 15,
    refresh_delay    = 0.09, # in sec
    res              = (900,1300),
    cor_func_scalar  = order_parameter,
    draw_arrows      = true,
    cor_func_vector  = angular_metric,   
    type_cor_func    = "vector",
    timesteps_corfunc = 100,
    );

### Initialize Interactive Simulation

This cell initializes the simulation state and visualization: it creates and fills the grids, builds the interactive agent list, prepares diffusion grids, configures Makie, opens the figure, and adds animation axes plus pause/parameter controls based on the parameter set defined above. 


In [ ]:
grids = create_grids(para)

grids, Full_agent_list = initialize_system(grids,para)

grids, Agent_list_cicular = make_sys_interactive!(grids,Full_agent_list)
diff_grids = create_diff_grids(para)

makie_config!(plot_para)

f = Figure(size = plot_para.res)
display(f)
ax1,ax2,fig_obs = ini_animation(Agent_list_cicular,para,grids,f,
    plot_para);

pause_button,para_obs = create_buttons(f,fig_obs,para, plot_para);

### Run/Pause Simulation Loop (Button-Controlled)

This cell attaches a click handler to the Play/Pause button and starts the simulation loop in an asynchronous task.  
The async loop is important because it lets the simulation run continuously **without freezing the Makie window** or blocking UI interactions.

When you press the button, the loop checks whether the figure window is still open, reads the current parameter values from the interactive controls, and then decides what to do based on the run state:

- If the simulation is paused, it waits briefly and checks again.
- If the simulation is running, it performs one diffusion update step.
- After a configured number of diffusion steps, it runs the agent-level updates (direction changes, movement, boundary handling, division, adsorption) and then redraws the plots.

So the simulation uses two update rates: frequent diffusion updates and less frequent agent/visualization updates.

**Output:** this cell itself does not print a final result to the notebook.  
Instead, the output is the **live animation in the figure window** (`ax1`, `ax2`), which updates over time while the simulation is running.

**How to use it:** after running this cell, you must **press the Play button** in the figure UI to start the simulation. Pressing it again pauses/stops the running loop state. Sometime you have to press the play button twice to start the simulation for the first time.

In [ ]:
on(pause_button.clicks) do shit
    i = 0
    @async while fig_obs.Isrunning[]
    # for unknown reasons loop does not work without @async,only use line below for debugging 
    #while fig_obs.Isrunning[]
        isopen(f.scene) || break 
        para = para_obs[]
        if fig_obs.Isrunning[] == false
            sleep(0.01)
            continue
        else
            diffusion_2D!(grids,diff_grids,para)
            i += 1

            if (i%para.pa_ph.ratio_walker_diff) == 0 
                update_directions!(grids, Agent_list_cicular, para)
                update_position!(grids, Agent_list_cicular, para)
                strenghten_boundary!(grids, para)
                divison!(grids, Agent_list_cicular,para)
                adsorb!(grids, Agent_list_cicular, para)
                animation_step!(ax1,ax2,Agent_list_cicular,grids,fig_obs, plot_para,para)
            end
        end
    end
end

### Record Simulation to Video

This cell exports the current simulation to an MP4 file by advancing the model state and rendering each frame to disk. It resets the time observable to start from a defined state, then records frames from step `2` to `para.timesteps` at the configured framerate.

Within each recorded frame, diffusion is applied multiple times (`ratio_walker_diff`) before one agent update cycle is performed. The agent cycle updates directions and positions, applies boundary strengthening, handles division and adsorption, and then redraws the figure via `animation_step!`.

**Output:** the main result is the saved video file in `vids/` (here: `TrypColonies_no_arrows_test_growth_1.mp4`). During recording, the Makie figure also updates in memory as frames are generated.

In [ ]:
fig_obs.Time[] = 1
record(f, "vids/TrypColonies_no_arrows_test_growth_1.mp4",  2:para.timesteps, framerate = plot_para.framerate) do i
    for t in 1:para.pa_ph.ratio_walker_diff
        diffusion_2D!(grids,diff_grids,para)
    end
    
    update_directions!(grids, Agent_list_cicular, para)
    update_position!(grids, Agent_list_cicular, para)
    strenghten_boundary!(grids, para)
    divison!(grids, Agent_list_cicular,para)
    adsorb!(grids, Agent_list_cicular, para)
    animation_step!(ax1,ax2,Agent_list_cicular,grids,fig_obs, plot_para,para)
end

### Parameter set 2
2. **Default parameters, reduced agent count** — the default physical parameter values from the paper on a 1000×1000 grid (15×15 mm), but with only 2,500 agents and a reduced boundary strength and bigger window size. This is a good compromise between biological realism and computational speed. Agent arrows are rendered.

In [ ]:
para_phys = parameters_physical(
    N                       = (1000,1000),
    L                       = (0.015,0.015),
    scale_fac               = 4,
    total_time              = 2000.0,
    walker_speed_phy        = 5*10^-6,
    Diff_coe                = 7*10^-11,
    walker_step_size        = 3,
    adsorption_rate         = 0.05,
    decay_rate              = 0.00001,
    Diameter_colony         = 0.003,
    agent_number            = 2500,
    noise_strength          = 0.3,
    grid_strength           = 150,  
    grid_recover_rate       = 15.0 /2.25,
    radius_collision        = 16,
    radius_tanget           = 15,
    Δt_walker_min           = 0.1, #alpha
    )

para = parameters( 
    pa_ph                   = para_phys,
    chemotaxis_flag         = true,
    repulsion_flag          = false,
    repulsion_range         = 2,
    geometry                = "circle",
    start_config_gradient   = "rand",
    )
    
plot_para = plot_parameters(
    arrow_tip_size   = 6,
    arrow_tail_length= 0.00005,
    framerate        = 30, 
    fontsize         = 15,
    refresh_delay    = 0.005,
    res              = (900,1300),
    cor_func_scalar  = order_parameter,
    draw_arrows      = true,
    cor_func_vector  = angular_metric,   
    type_cor_func    = "vector",
    timesteps_corfunc = 100,
    );

### Initialize Interactive Simulation

This cell initializes the simulation state and visualization: it creates and fills the grids, builds the interactive agent list, prepares diffusion grids, configures Makie, opens the figure, and adds animation axes plus pause/parameter controls based on the parameter set defined above. 

In [ ]:
grids = create_grids(para)

grids, Full_agent_list = initialize_system(grids,para)

grids, Agent_list_cicular = make_sys_interactive!(grids,Full_agent_list)
diff_grids = create_diff_grids(para)

makie_config!(plot_para)

f = Figure(size = plot_para.res)
display(f)
ax1,ax2,fig_obs = ini_animation(Agent_list_cicular,para,grids,f,
    plot_para);

pause_button,para_obs = create_buttons(f,fig_obs,para, plot_para);

### Run/Pause Simulation Loop (Button-Controlled)

This cell attaches a click handler to the Play/Pause button and starts the simulation loop in an asynchronous task.  
The async loop is important because it lets the simulation run continuously **without freezing the Makie window** or blocking UI interactions.

When you press the button, the loop checks whether the figure window is still open, reads the current parameter values from the interactive controls, and then decides what to do based on the run state:

- If the simulation is paused, it waits briefly and checks again.
- If the simulation is running, it performs one diffusion update step.
- After a configured number of diffusion steps, it runs the agent-level updates (direction changes, movement, boundary handling, division, adsorption) and then redraws the plots.

So the simulation uses two update rates: frequent diffusion updates and less frequent agent/visualization updates.

**Output:** this cell itself does not print a final result to the notebook.  
Instead, the output is the **live animation in the figure window** (`ax1`, `ax2`), which updates over time while the simulation is running.

**How to use it:** after running this cell, you must **press the Play button** in the figure UI to start the simulation. Pressing it again pauses/stops the running loop state. Sometime you have to press the play button twice to start the simulation for the first time.

In [ ]:
on(pause_button.clicks) do shit
    i = 0
    @async while fig_obs.Isrunning[]
    # for unknown reasons loop does not work without @async,only use line below for debugging 
    #while fig_obs.Isrunning[]
        isopen(f.scene) || break 
        para = para_obs[]
        if fig_obs.Isrunning[] == false
            sleep(0.01)
            continue
        else
            diffusion_2D!(grids,diff_grids,para)
            i += 1

            if (i%para.pa_ph.ratio_walker_diff) == 0 
                update_directions!(grids, Agent_list_cicular, para)
                update_position!(grids, Agent_list_cicular, para)
                strenghten_boundary!(grids, para)
                divison!(grids, Agent_list_cicular,para)
                adsorb!(grids, Agent_list_cicular, para)
                animation_step!(ax1,ax2,Agent_list_cicular,grids,fig_obs, plot_para,para)
            end
        end
    end
end

### Record Simulation to Video

This cell exports the current simulation to an MP4 file by advancing the model state and rendering each frame to disk. It resets the time observable to start from a defined state, then records frames from step `2` to `para.timesteps` at the configured framerate.

Within each recorded frame, diffusion is applied multiple times (`ratio_walker_diff`) before one agent update cycle is performed. The agent cycle updates directions and positions, applies boundary strengthening, handles division and adsorption, and then redraws the figure via `animation_step!`.

**Output:** the main result is the saved video file in `vids/` (here: `TrypColonies_no_arrows_test_growth_2.mp4`). During recording, the Makie figure also updates in memory as frames are generated.

In [ ]:
fig_obs.Time[] = 1
record(f, "vids/TrypColonies_no_arrows_test_growth_2.mp4",  2:para.timesteps, framerate = plot_para.framerate) do i
    for t in 1:para.pa_ph.ratio_walker_diff
        diffusion_2D!(grids,diff_grids,para)
    end
    
    update_directions!(grids, Agent_list_cicular, para)
    update_position!(grids, Agent_list_cicular, para)
    strenghten_boundary!(grids, para)
    divison!(grids, Agent_list_cicular,para)
    adsorb!(grids, Agent_list_cicular, para)
    animation_step!(ax1,ax2,Agent_list_cicular,grids,fig_obs, plot_para,para)
end

### Parameter set 3
3. **Full default parameters** — the complete default parameter set from the paper: 250,000 agents on a 1000×1000 grid (15×15 mm) at biologically realistic speed (5×10⁻⁶ m/s). **Note:** this configuration is computationally demanding. Even with a multi-threaded Julia kernel, expect approximately real-time performance on hardware available in 2025 (i.e. roughly one simulation second per wall-clock second). Agent arrows are disabled (`draw_arrows = false`) as individual agents cannot be distinguished at this density. Use as many threads as you can spare, you can do this for example by creating a custom Julia kernel  more threads (in this case 8):

In [ ]:
para_phys = parameters_physical(
    N                       = (1000,1000),
    L                       = (0.015,0.015),
    scale_fac               = 4,
    total_time              = 600.0,
    walker_speed_phy        = 5*10^-6,
    Diff_coe                = 7*10^-11,
    walker_step_size        = 3,
    adsorption_rate         = 1*10^-6,
    decay_rate              = 0.0001,
    Diameter_colony         = 0.003,
    agent_number            = 250000,
    noise_strength          = 0.3,
    grid_strength           = 8000,  
    grid_recover_rate       = 15.0 /2.25,
    radius_collision        = 12,
    radius_tanget           = 15,
    Δt_walker_min           = 0.1, # alpha 
    )

para = parameters( 
    pa_ph                   = para_phys,
    chemotaxis_flag         = true,
    repulsion_flag          = false,
    repulsion_range         = 2,
    geometry                = "circle",
    start_config_gradient   = "rand",
    )
    
plot_para = plot_parameters(
    arrow_tip_size   = 6,
    arrow_tail_length= 0.00005,
    framerate        = 30, 
    fontsize         = 15,
    refresh_delay    = 0.0005,
    res              = (1200,1800),
    cor_func_scalar  = order_parameter,
    draw_arrows      = false,
    cor_func_vector  = angular_metric,   
    type_cor_func    = "vector",
    timesteps_corfunc = 100,
    );

In [ ]:
grids = create_grids(para)

grids, Full_agent_list = initialize_system(grids,para)

grids, Agent_list_cicular = make_sys_interactive!(grids,Full_agent_list)
diff_grids = create_diff_grids(para)

makie_config!(plot_para)

f = Figure(size = plot_para.res)
display(f)
ax1,ax2,fig_obs = ini_animation(Agent_list_cicular,para,grids,f,
    plot_para);

pause_button,para_obs = create_buttons(f,fig_obs,para, plot_para);

In [ ]:
on(pause_button.clicks) do shit
    i = 0
    @async while fig_obs.Isrunning[]
    # for unknown reasons loop does not work without @async,only use line below for debugging 
    #while fig_obs.Isrunning[]
        isopen(f.scene) || break 
        para = para_obs[]
        if fig_obs.Isrunning[] == false
            sleep(0.01)
            continue
        else
            diffusion_2D!(grids,diff_grids,para)
            i += 1

            if (i%para.pa_ph.ratio_walker_diff) == 0 
                update_directions!(grids, Agent_list_cicular, para)
                update_position!(grids, Agent_list_cicular, para)
                strenghten_boundary!(grids, para)
                divison!(grids, Agent_list_cicular,para)
                adsorb!(grids, Agent_list_cicular, para)
                animation_step!(ax1,ax2,Agent_list_cicular,grids,fig_obs, plot_para,para)
            end
        end
    end
end

In [ ]:
fig_obs.Time[] = 1
record(f, "vids/TrypColonies_no_arrows_test_growth_2.mp4",  2:para.timesteps, framerate = plot_para.framerate) do i
    for t in 1:para.pa_ph.ratio_walker_diff
        diffusion_2D!(grids,diff_grids,para)
    end
    
    update_directions!(grids, Agent_list_cicular, para)
    update_position!(grids, Agent_list_cicular, para)
    strenghten_boundary!(grids, para)
    divison!(grids, Agent_list_cicular,para)
    adsorb!(grids, Agent_list_cicular, para)
    animation_step!(ax1,ax2,Agent_list_cicular,grids,fig_obs, plot_para,para)
end